# Подготовка данных

Пайплайн: Excel → CSV → кроп → склейка → тайлинг → стратифицированный сплит по скважинам.

In [ ]:
# ── Импорты ─────────────────────────────────────────────────────────────────
import os
import re
import sys
import json
import shutil
import random
from pathlib import Path
from collections import Counter, defaultdict
from typing import Dict, Optional, Tuple, List

# Добавляем корень проекта в путь для импорта config.py
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm import tqdm

from config import (
    DIGITAL_CORE as DATA_ROOT,
    PIPELINE_CSV  as CSV_ROOT,
    PIPELINE_TILES as TILE_ROOT,
    DATASET_ROOT  as DS_ROOT,
    DATASET_STATS,
    TILE_CM, OVERLAP_CM,
    VAL_FRAC, TEST_FRAC, SEED,
    MODALITIES,
    CLASS_PALETTE, CLASS_SHORT,
)

# ── Параметры тайлинга ────────────────────────────────────────────────────
TILE_MIN_H = 20      # минимальная высота тайла в пикселях

# ── Прочее ──────────────────────────────────────────────────────────────
IMG_EXTS   = {'.jpg', '.jpeg', '.png', '.bmp'}

# ── Маппинг классов ──────────────────────────────────────────────────────
CLASS_RULES: Dict[str, Optional[str]] = {
    # УДАЛЯЕМ
    'Алевролит_песчанистый':                                    None,
    'Песчаник_с_включениями_угля':                              None,
    'Известняк':                                                None,
    'Породы_фундамента':                                        None,

    # АРГИЛЛИТ (класс Уголь влит сюда — все варианты мусорной метки)
    'Аргиллит':                                                 'Аргиллит',
    'Аргиллит_углистый':                                        'Аргиллит',
    'Аргиллит_с_включениями_угля':                              'Аргиллит',
    'Уголь':                                                    'Аргиллит',
    'Уголь_с_прослоями_аргиллита':                              'Аргиллит',

    # АЛЕВРОЛИТ
    'Алевролит':                                                'Алевролит',
    'Алевролит_с_включениями_угля':                             'Алевролит',
    'Алевролит_глинистый':                                      'Алевролит',
    'Алевролит_карбонатный':                                    'Алевролит',

    # ПЕРЕСЛАИВАНИЕ АРГИЛЛИТА И АЛЕВРОЛИТА
    'Переслаивание_аргиллита_и_алевролита':                     'Переслаивание_аргиллита_и_алевролита',
    'Аргиллит_алевритовый':                                     'Переслаивание_аргиллита_и_алевролита',
    'Алевролит_с_прослоями_аргиллита':                          'Переслаивание_аргиллита_и_алевролита',
    'Аргиллит_с_прослоями_алевролита':                          'Переслаивание_аргиллита_и_алевролита',

    # ПЕРЕСЛАИВАНИЕ ВСЕХ ТРЁХ
    'Переслаивание_песчаника,_аргиллита_и_алевролита':          'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Песчаник_с_включениями_алевролита_и_аргиллита':            'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Чередование_аргиллита,_алевролита_и_песчаника':            'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Алевролит_с_прослоями_песчаника_и_аргиллита':              'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Песчаник_с_прослоями_алевролита_и_аргиллита':              'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Аргиллит_с_прослоями_песчаника_и_алевролита':              'Переслаивание_песчаника,_аргиллита_и_алевролита',

    # ГЛИНИСТО-КАРБОНАТНАЯ
    'Глинисто-карбонатная_порода':                              'Глинисто-карбонатная_порода',
    'Опока_глинистая':                                          'Глинисто-карбонатная_порода',
    'Глина_опоковидная':                                        'Глинисто-карбонатная_порода',
    'Глина_аргиллитоподобная_с_прослоями_глины_опоковидной':    'Глинисто-карбонатная_порода',
    'Кремнисто-глинистая_порода':                               'Глинисто-карбонатная_порода',
    'Глина_опоковидная_с_включением_глинистых_опок':            'Глинисто-карбонатная_порода',
    'Глина_аргиллитоподобная':                                  'Глинисто-карбонатная_порода',

    # ПЕСЧАНИК С ПРОСЛОЯМИ АЛЕВРОЛИТА
    'Песчаник_с_прослоями_алевролита':                          'Песчаник_с_прослоями_алевролита',
    'Переслаивание_песчаника_и_алевролита':                     'Песчаник_с_прослоями_алевролита',
    'Алевролит_с_прослоями_песчаника':                          'Песчаник_с_прослоями_алевролита',

    # ПЕСЧАНИК С ПРОСЛОЯМИ АРГИЛЛИТА
    'Песчаник_с_прослоями_аргиллита':                           'Песчаник_с_прослоями_аргиллита',
    'Аргиллит_с_прослоями_песчаника':                           'Песчаник_с_прослоями_аргиллита',
    'Переслаивание_песчаника_и_аргиллита':                      'Песчаник_с_прослоями_аргиллита',

    # ПЕСЧАНИК
    'Песчаник':                                                 'Песчаник',
    'Песчаник_карбонатный':                                     'Песчаник',
}

CLASSES_ORDER = [
    'Песчаник',
    'Аргиллит',
    'Алевролит',
    'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Переслаивание_аргиллита_и_алевролита',
    'Песчаник_с_прослоями_алевролита',
    'Песчаник_с_прослоями_аргиллита',
    'Глинисто-карбонатная_порода',
]

# ── Вспомогательные функции ──────────────────────────────────────────────
def load_image(path: Path) -> np.ndarray:
    with Image.open(path) as img:
        return cv2.cvtColor(np.array(img.convert('RGB')), cv2.COLOR_RGB2BGR)


def save_image(path: Path, img: np.ndarray) -> None:
    rgb = cv2.cvtColor(np.clip(img, 0, 255).astype(np.uint8), cv2.COLOR_BGR2RGB)
    Image.fromarray(rgb).save(str(path), quality=90)


def norm_mineral(name: str) -> str:
    """Excel mineral name → CLASS_RULES key."""
    return name.strip().replace(' ', '_')


print('Конфигурация загружена.')
print(f'  DATA_ROOT  : {DATA_ROOT.resolve()}')
print(f'  CSV_ROOT   : {CSV_ROOT.resolve()}')
print(f'  TILE_ROOT  : {TILE_ROOT.resolve()}')
print(f'  DS_ROOT    : {DS_ROOT.resolve()}')
print(f'  STATS_DIR  : {DATASET_STATS.resolve()}')
print(f'  Классов    : {len(CLASSES_ORDER)}')

## Stage 1 — Excel → CSV

In [4]:
def stage1_excel_to_csv():
    CSV_ROOT.mkdir(parents=True, exist_ok=True)
    wells = sorted(d for d in DATA_ROOT.iterdir() if d.is_dir())
    print(f'Найдено скважин: {len(wells)}')

    ok = skip = fail = 0
    for well_dir in wells:
        well     = well_dir.name
        csv_path = CSV_ROOT / f'{well}.csv'

        if csv_path.exists():
            skip += 1
            continue

        xlsx_files = [f for f in well_dir.glob('*.xlsx') if not f.name.startswith('~$')]
        if not xlsx_files:
            print(f'  [SKIP] {well}: Excel не найден')
            fail += 1
            continue

        try:
            raw = pd.read_excel(xlsx_files[0], usecols='B:G', skiprows=8, header=None)
            raw = raw.drop(5, axis=1)
            for col in [1, 3, 4]:
                raw[col] = pd.to_numeric(raw[col], errors='coerce')
            raw = raw.dropna(subset=[1, 3, 4])
            raw[2] = raw[1] + raw[4]
            raw[1] = raw[1] + raw[3]
            raw = raw.drop([3, 4], axis=1).dropna()
            raw = raw.rename(columns={1: 'depth_from', 2: 'depth_to', 6: 'mineral'})
            raw['mineral'] = raw['mineral'].astype(str).str.strip()
            raw.to_csv(csv_path, index=False, encoding='utf-8-sig')
            print(f'  [OK]   {well}: {len(raw)} строк → {csv_path.name}')
            ok += 1
        except Exception as e:
            print(f'  [ERR]  {well}: {e}')
            fail += 1

    print(f'\nCSV готово: создано={ok}  пропущено(кеш)={skip}  ошибок={fail}')


stage1_excel_to_csv()

Найдено скважин: 16


/var/lib/sadmin/Рабочий стол/kern/venv/lib64/python3.11/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: Посл_опис_керна!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/var/lib/sadmin/Рабочий стол/kern/venv/lib64/python3.11/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: Посл_опис_керна!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


  [OK]   Восточно-Падинск_3-ВП: 332 строк → Восточно-Падинск_3-ВП.csv
  [OK]   Соболох-Неджелинск_3: 213 строк → Соболох-Неджелинск_3.csv
  [OK]   Таб-Яхинская_10360: 12 строк → Таб-Яхинская_10360.csv
  [OK]   Харасавэйск_1100: 39 строк → Харасавэйск_1100.csv
  [OK]   Харасавэйск_1400: 35 строк → Харасавэйск_1400.csv


/var/lib/sadmin/Рабочий стол/kern/venv/lib64/python3.11/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: Посл_опис_керна!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/var/lib/sadmin/Рабочий стол/kern/venv/lib64/python3.11/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: Посл_опис_керна!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/var/lib/sadmin/Рабочий стол/kern/venv/lib64/python3.11/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: Посл_опис_керна!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/var/lib/sadmin/Рабочий стол/kern/venv/lib64/python3.11/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: Посл_опис_керна!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


  [OK]   Харасавэйск_1700: 68 строк → Харасавэйск_1700.csv
  [OK]   Харасавэйск_1800: 43 строк → Харасавэйск_1800.csv
  [OK]   Харасавэйск_1900: 53 строк → Харасавэйск_1900.csv
  [OK]   Харасавэйск_2000: 81 строк → Харасавэйск_2000.csv
  [OK]   Харасавэйск_300: 39 строк → Харасавэйск_300.csv


/var/lib/sadmin/Рабочий стол/kern/venv/lib64/python3.11/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: Посл_опис_керна!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/var/lib/sadmin/Рабочий стол/kern/venv/lib64/python3.11/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: Посл_опис_керна!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/var/lib/sadmin/Рабочий стол/kern/venv/lib64/python3.11/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: Посл_опис_керна!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/var/lib/sadmin/Рабочий стол/kern/venv/lib64/python3.11/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: Посл_опис_керна!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


  [OK]   Харасавэйск_600 к 6: 56 строк → Харасавэйск_600 к 6.csv
  [OK]   Харасавэйск_700: 96 строк → Харасавэйск_700.csv
  [OK]   Харасавэйск_900: 57 строк → Харасавэйск_900.csv
  [OK]   Ю-Песцовый лу_12: 278 строк → Ю-Песцовый лу_12.csv
  [OK]   Ямбург_1 N: 158 строк → Ямбург_1 N.csv
  [OK]   Ямбургск_24604: 68 строк → Ямбургск_24604.csv

CSV готово: создано=16  пропущено(кеш)=0  ошибок=0


/var/lib/sadmin/Рабочий стол/kern/venv/lib64/python3.11/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: Посл_опис_керна!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/var/lib/sadmin/Рабочий стол/kern/venv/lib64/python3.11/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: Посл_опис_керна!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/var/lib/sadmin/Рабочий стол/kern/venv/lib64/python3.11/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: Посл_опис_керна!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/var/lib/sadmin/Рабочий стол/kern/venv/lib64/python3.11/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: Посл_опис_керна!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


## Stage 2 — Кроп + Склейка + Тайлинг

Объединяет три отдельных шага старого пайплайна в один проход без промежуточных директорий.

In [5]:
# Парсинг глубин из имени фото: "..._1234.567 - 2345.678" или "..._1234.567-2345.678"
_DEPTH_RE = re.compile(r'_([\d]+[.,][\d]+)\s*-\s*([\d]+[.,][\d]+)$')


def parse_photo_depth(stem: str) -> Optional[Tuple[float, float]]:
    m = _DEPTH_RE.search(stem)
    if not m:
        return None
    return float(m.group(1).replace(',', '.')), float(m.group(2).replace(',', '.'))


def process_well(well_dir: Path, csv_path: Path, out_root: Path) -> Tuple[int, set]:
    """
    Для одной скважины: читает CSV → для каждого интервала находит фото →
    кропит → склеивает → нарезает на тайлы.
    Возвращает (кол-во тайлов, множество неизвестных минералов).
    """
    df = pd.read_csv(csv_path, encoding='utf-8-sig')

    # Ищем папку ФОТО (без учёта регистра)
    photo_dir = next(
        (c for c in well_dir.iterdir() if c.is_dir() and c.name.lower() == 'фото'),
        None
    )
    if photo_dir is None:
        return 0, set()

    total       = 0
    unknown_set = set()

    for modality in MODALITIES:
        mod_dir = photo_dir / modality
        if not mod_dir.exists():
            continue

        # Индексируем все фото: список (depth_from, depth_to, path)
        photos: List[Tuple[float, float, Path]] = []
        for img_path in sorted(mod_dir.iterdir()):
            if img_path.suffix.lower() not in IMG_EXTS:
                continue
            depths = parse_photo_depth(img_path.stem)
            if depths is None:
                continue
            photos.append((depths[0], depths[1], img_path))

        if not photos:
            continue

        # Обрабатываем каждый интервал из CSV
        for _, row in df.iterrows():
            mineral_key = norm_mineral(str(row['mineral']))

            if mineral_key not in CLASS_RULES:
                unknown_set.add(mineral_key)
                continue

            mapped_class = CLASS_RULES[mineral_key]
            if mapped_class is None:
                continue

            d_from = float(row['depth_from'])
            d_to   = float(row['depth_to'])
            if d_to <= d_from:
                continue

            # Находим перекрывающиеся фото
            overlaps = [
                (pf, pt, pp) for pf, pt, pp in photos
                if pf < d_to and pt > d_from
            ]
            if not overlaps:
                continue

            # Кропим каждое фото по пересечению с [d_from, d_to]
            crops: List[np.ndarray] = []
            for pf, pt, pp in sorted(overlaps, key=lambda x: x[0]):
                inter_start = max(d_from, pf)
                inter_end   = min(d_to, pt)
                if inter_start >= inter_end:
                    continue
                try:
                    img = load_image(pp)
                except Exception:
                    continue
                h      = img.shape[0]
                ph_cm  = (pt - pf) * 100.0
                if ph_cm <= 0:
                    continue
                ppc    = h / ph_cm
                y0     = max(0, int((inter_start - pf) * 100.0 * ppc))
                y1     = min(h, int(round((inter_end - pf) * 100.0 * ppc)))
                if y0 >= y1:
                    continue
                crops.append(img[y0:y1, :])

            if not crops:
                continue

            # Склеиваем кропы вертикально (выравниваем ширину)
            max_w  = max(c.shape[1] for c in crops)
            padded = []
            for c in crops:
                if c.shape[1] < max_w:
                    c = cv2.copyMakeBorder(c, 0, 0, 0, max_w - c.shape[1],
                                           cv2.BORDER_CONSTANT, value=0)
                padded.append(c)
            merged = np.vstack(padded)

            # Нарезаем на тайлы
            h_total  = merged.shape[0]
            total_cm = (d_to - d_from) * 100.0
            ppc      = h_total / total_cm
            tile_px  = max(1, int(round(TILE_CM * ppc)))
            step_px  = max(1, int(round((TILE_CM - OVERLAP_CM) * ppc)))

            dst_dir = out_root / well_dir.name / modality / mapped_class
            dst_dir.mkdir(parents=True, exist_ok=True)

            frag       = 0
            current_cm = 0.0
            while current_cm + TILE_CM <= total_cm + 1e-6:
                y0 = int(round(current_cm * ppc))
                y1 = y0 + tile_px
                if y1 > h_total:
                    break
                tile = merged[y0:y1, :]
                if tile.shape[0] >= TILE_MIN_H:
                    fname = f'{mineral_key}_{d_from:.3f}_{d_to:.3f}_frag{frag:04d}.jpg'
                    save_image(dst_dir / fname, tile)
                    frag  += 1
                    total += 1
                current_cm += (TILE_CM - OVERLAP_CM)

    return total, unknown_set


def stage2_process_all_wells():
    TILE_ROOT.mkdir(parents=True, exist_ok=True)
    wells = sorted(d for d in DATA_ROOT.iterdir() if d.is_dir())

    grand_total = 0
    all_unknown: set = set()

    for well_dir in tqdm(wells, desc='Скважины'):
        csv_path = CSV_ROOT / f'{well_dir.name}.csv'
        if not csv_path.exists():
            print(f'  [SKIP] {well_dir.name}: CSV не найден')
            continue

        n, unknown = process_well(well_dir, csv_path, TILE_ROOT)
        grand_total += n
        if unknown:
            print(f'  [WARN] {well_dir.name}: неизвестные минералы: {unknown}')
            all_unknown |= unknown

    print(f'\nВсего тайлов: {grand_total:,}')
    if all_unknown:
        print(f'Добавьте в CLASS_RULES: {sorted(all_unknown)}')


stage2_process_all_wells()

Скважины: 100%|█████████████████████████████████| 16/16 [01:18<00:00,  4.89s/it]


Всего тайлов: 97,494


## Stage 3 — Сканирование тайлов

In [6]:
def scan_tiles(tile_root: Path) -> Dict:
    """
    Возвращает well_class_counts[well][modality][class] = count.
    """
    counts: Dict = defaultdict(lambda: defaultdict(Counter))
    for well_dir in sorted(tile_root.iterdir()):
        if not well_dir.is_dir():
            continue
        for mod_dir in well_dir.iterdir():
            if not mod_dir.is_dir() or mod_dir.name not in MODALITIES:
                continue
            for cls_dir in mod_dir.iterdir():
                if not cls_dir.is_dir():
                    continue
                n = sum(1 for f in cls_dir.iterdir() if f.suffix.lower() == '.jpg')
                if n > 0:
                    counts[well_dir.name][mod_dir.name][cls_dir.name] = n
    return dict(counts)


well_class_counts = scan_tiles(TILE_ROOT)
print(f'Скважин с тайлами: {len(well_class_counts)}')

# Агрегируем по всем скважинам и модальностям
class_totals: Counter = Counter()
for mods in well_class_counts.values():
    for cls_counts in mods.values():
        class_totals.update(cls_counts)

print(f'\n{"Класс":<55} {"Итого":>8}')
print('-' * 65)
for cls in CLASSES_ORDER:
    print(f'  {cls:<53} {class_totals.get(cls, 0):>8,}')

unknown_classes = set(class_totals) - set(CLASSES_ORDER)
if unknown_classes:
    print(f'\nКлассы вне CLASSES_ORDER: {unknown_classes}')

Скважин с тайлами: 16

Класс                                                      Итого
-----------------------------------------------------------------
  Песчаник                                                31,996
  Аргиллит                                                 9,348
  Алевролит                                                6,496
  Переслаивание_песчаника,_аргиллита_и_алевролита         14,406
  Переслаивание_аргиллита_и_алевролита                     9,500
  Песчаник_с_прослоями_алевролита                         15,472
  Песчаник_с_прослоями_аргиллита                           3,502
  Глинисто-карбонатная_порода                              6,774


## Stage 4 — Стратифицированный сплит по скважинам

Жадный алгоритм: назначает скважины в val и test так, чтобы каждый класс имел ~10% в каждом сплите.

In [7]:
def _well_totals(well_class_counts: Dict) -> Dict[str, Counter]:
    """Суммируем тайлы по всем модальностям для каждой скважины."""
    result = {}
    for well, mods in well_class_counts.items():
        c: Counter = Counter()
        for cls_counts in mods.values():
            c.update(cls_counts)
        result[well] = c
    return result


def stratified_well_split(
    well_class_counts: Dict,
    val_frac: float = VAL_FRAC,
    test_frac: float = TEST_FRAC,
    seed: int = SEED,
) -> Dict[str, str]:
    """
    Жадно назначает каждую скважину в train/val/test.
    Цель: минимизировать среднеквадратичное отклонение доли каждого класса
    от целевых val_frac / test_frac.
    """
    rng = random.Random(seed)

    wt           = _well_totals(well_class_counts)
    global_total: Counter = Counter()
    for c in wt.values():
        global_total.update(c)

    all_classes  = list(global_total.keys())
    wells        = list(wt.keys())
    rng.shuffle(wells)          # воспроизводимая случайность перед жадным шагом

    split_counts: Dict[str, Counter] = {
        'train': Counter(), 'val': Counter(), 'test': Counter()
    }
    well_to_split: Dict[str, str] = {}
    unassigned = list(wells)

    def _assign(target_split: str, target_frac: float) -> None:
        while unassigned:
            # Остановиться, если доля уже достигнута
            assigned = sum(split_counts[target_split].values())
            grand    = sum(global_total.values())
            if grand > 0 and assigned / grand >= target_frac:
                break

            # Выбрать скважину, минимизирующую отклонение от target_frac
            def _score(w: str) -> float:
                trial = Counter(split_counts[target_split])
                trial.update(wt[w])
                return sum(
                    (trial[c] / max(global_total[c], 1) - target_frac) ** 2
                    for c in all_classes
                )

            best = min(unassigned, key=_score)
            unassigned.remove(best)
            well_to_split[best] = target_split
            split_counts[target_split].update(wt[best])

    _assign('val',  val_frac)
    _assign('test', test_frac)

    for w in unassigned:
        well_to_split[w] = 'train'
        split_counts['train'].update(wt[w])

    return well_to_split, split_counts


well_to_split, split_counts = stratified_well_split(well_class_counts)

# Вывод назначения
print('Назначение скважин:')
for split in ['train', 'val', 'test']:
    ws = sorted(w for w, s in well_to_split.items() if s == split)
    print(f'  {split:5}: {ws}')

# Вывод баланса классов
print(f'\n{"Класс":<55} {"Train":>8} {"Val":>6} {"Test":>6} {"Val%":>6}')
print('-' * 85)
for cls in CLASSES_ORDER:
    tr    = split_counts['train'].get(cls, 0)
    vl    = split_counts['val'].get(cls, 0)
    te    = split_counts['test'].get(cls, 0)
    total = tr + vl + te
    vpct  = vl / total * 100 if total > 0 else 0
    flag  = '  !' if (vpct < 5 or vpct > 30) else ''
    print(f'  {cls:<53} {tr:>8,} {vl:>6,} {te:>6,} {vpct:>5.1f}%{flag}')

print()
bad = [c for c in CLASSES_ORDER
       if split_counts['val'].get(c, 0) == 0 or split_counts['test'].get(c, 0) == 0]
if bad:
    print(f'ВНИМАНИЕ: классы без val/test: {bad}')
else:
    print('Все классы присутствуют в train/val/test.')

Назначение скважин:
  train: ['Восточно-Падинск_3-ВП', 'Соболох-Неджелинск_3', 'Таб-Яхинская_10360', 'Харасавэйск_1700', 'Харасавэйск_1800', 'Харасавэйск_1900', 'Харасавэйск_300', 'Харасавэйск_600 к 6', 'Харасавэйск_900', 'Ю-Песцовый лу_12', 'Ямбургск_24604']
  val  : ['Харасавэйск_1100', 'Харасавэйск_2000', 'Харасавэйск_700']
  test : ['Харасавэйск_1400', 'Ямбург_1 N']

Класс                                                      Train    Val   Test   Val%
-------------------------------------------------------------------------------------
  Песчаник                                                23,822  4,676  3,498  14.6%
  Аргиллит                                                 6,900  1,154  1,294  12.3%
  Алевролит                                                4,900    920    676  14.2%
  Переслаивание_песчаника,_аргиллита_и_алевролита         11,146  1,678  1,582  11.6%
  Переслаивание_аргиллита_и_алевролита                     6,796  1,360  1,344  14.3%
  Песчаник_с_прослоями_а

## Stage 5 — Копирование в финальный датасет

In [8]:
def stage5_build_dataset():
    DS_ROOT.mkdir(parents=True, exist_ok=True)

    total_copied = 0
    manifest: Dict = {}

    for well, split in tqdm(well_to_split.items(), desc='Копирование'):
        manifest[well] = {'split': split, 'classes': {}}
        well_dir = TILE_ROOT / well
        if not well_dir.exists():
            continue

        for mod_dir in well_dir.iterdir():
            if not mod_dir.is_dir() or mod_dir.name not in MODALITIES:
                continue
            for cls_dir in mod_dir.iterdir():
                if not cls_dir.is_dir():
                    continue
                dst = DS_ROOT / mod_dir.name / split / cls_dir.name
                dst.mkdir(parents=True, exist_ok=True)
                n = 0
                for src_file in cls_dir.iterdir():
                    if src_file.suffix.lower() == '.jpg':
                        shutil.copy2(src_file, dst / src_file.name)
                        n += 1
                        total_copied += 1
                key = f'{mod_dir.name}/{cls_dir.name}'
                manifest[well]['classes'][key] = \
                    manifest[well]['classes'].get(key, 0) + n

    # label_encoder.json
    class_to_idx = {cls: idx for idx, cls in enumerate(CLASSES_ORDER)}
    with open(DS_ROOT / 'label_encoder.json', 'w', encoding='utf-8') as f:
        json.dump(class_to_idx, f, ensure_ascii=False, indent=2)

    # split_manifest.json
    with open(DS_ROOT / 'split_manifest.json', 'w', encoding='utf-8') as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    print(f'Скопировано файлов: {total_copied:,}')
    print(f'Датасет: {DS_ROOT.resolve()}')
    print('  label_encoder.json  сохранён')
    print('  split_manifest.json сохранён')


stage5_build_dataset()

Копирование: 100%|██████████████████████████████| 16/16 [00:26<00:00,  1.67s/it]

Скопировано файлов: 97,494
Датасет: /var/lib/sadmin/Рабочий стол/kern/ML-Core-Kern-/data/dataset
  label_encoder.json  сохранён
  split_manifest.json сохранён


## Stage 6 — Итоговая статистика

In [9]:
def compute_final_stats() -> Dict:
    """Подсчёт тайлов по modality / split / class в DS_ROOT."""
    counts: Dict = defaultdict(lambda: defaultdict(Counter))
    for mod in MODALITIES:
        mod_dir = DS_ROOT / mod
        if not mod_dir.exists():
            continue
        for split_dir in mod_dir.iterdir():
            if not split_dir.is_dir():
                continue
            for cls_dir in split_dir.iterdir():
                if not cls_dir.is_dir():
                    continue
                n = sum(1 for f in cls_dir.iterdir() if f.suffix.lower() == '.jpg')
                counts[mod][split_dir.name][cls_dir.name] = n
    return dict(counts)


final_stats = compute_final_stats()

for mod in MODALITIES:
    mst = final_stats.get(mod, {})
    print(f'\n{"="*75}')
    print(f'  Модальность: {mod}')
    print(f'{"="*75}')
    print(f'  {"Класс":<53} {"Train":>8} {"Val":>6} {"Test":>6}')
    print(f'  {"-"*71}')
    for cls in CLASSES_ORDER:
        tr = mst.get('train', {}).get(cls, 0)
        vl = mst.get('val',   {}).get(cls, 0)
        te = mst.get('test',  {}).get(cls, 0)
        print(f'  {cls:<53} {tr:>8,} {vl:>6,} {te:>6,}')
    print(f'  {"-"*71}')
    all_tr = sum(mst.get('train', {}).values())
    all_vl = sum(mst.get('val',   {}).values())
    all_te = sum(mst.get('test',  {}).values())
    print(f'  {"ИТОГО":<53} {all_tr:>8,} {all_vl:>6,} {all_te:>6,}')

    # Дисбаланс
    tr_vals = [v for v in mst.get('train', {}).values() if v > 0]
    if tr_vals:
        mx, mn = max(tr_vals), min(tr_vals)
        print(f'\n  Дисбаланс train: {mx / max(mn, 1):.1f}:1  (макс={mx:,} мин={mn:,})')

    # Проверка полноты
    missing_val  = [c for c in CLASSES_ORDER if mst.get('val',  {}).get(c, 0) == 0]
    missing_test = [c for c in CLASSES_ORDER if mst.get('test', {}).get(c, 0) == 0]
    if missing_val or missing_test:
        if missing_val:  print(f'  ВНИМАНИЕ: нет в val:  {missing_val}')
        if missing_test: print(f'  ВНИМАНИЕ: нет в test: {missing_test}')
    else:
        print(f'  Все {len(CLASSES_ORDER)} классов присутствуют в train/val/test.')


  Модальность: ДС
  Класс                                                    Train    Val   Test
  -----------------------------------------------------------------------
  Песчаник                                                11,911  2,338  1,749
  Аргиллит                                                 3,450    577    647
  Алевролит                                                2,450    460    338
  Переслаивание_песчаника,_аргиллита_и_алевролита          5,573    839    791
  Переслаивание_аргиллита_и_алевролита                     3,398    680    672
  Песчаник_с_прослоями_алевролита                          5,594    991  1,151
  Песчаник_с_прослоями_аргиллита                           1,393    297     61
  Глинисто-карбонатная_порода                              2,661    679     47
  -----------------------------------------------------------------------
  ИТОГО                                                   36,430  6,861  5,456

  Дисбаланс train: 8.6:1  (макс=11,911 мин

## Stage 7 — Визуальные артефакты датасета для отчёта

Сохраняются в `results/dataset_stats/`:
- `class_distribution.png` — гистограмма классов по сплитам
- `split_table.png` — таблица counts + % 
- `tile_examples.png` — сетка примеров тайлов по классам

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker

DATASET_STATS.mkdir(parents=True, exist_ok=True)

# ── Вспомогательные данные ────────────────────────────────────────────────────
# Переиспользуем final_stats из Stage 6 (modality ДС достаточно — цифры идентичны)
mst = final_stats.get('ДС', {})
splits = ['train', 'val', 'test']
split_colors = {'train': '#3B82F6', 'val': '#F59E0B', 'test': '#EF4444'}
short_names = [CLASS_SHORT.get(c, c[:10]) for c in CLASSES_ORDER]
colors_by_class = [CLASS_PALETTE.get(c, '#94A3B8') for c in CLASSES_ORDER]

# ═══════════════════════════════════════════════════════════════════════════════
# 1. class_distribution.png — гистограмма: количество тайлов по классам × сплитам
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 6))

n_cls = len(CLASSES_ORDER)
x = np.arange(n_cls)
width = 0.25

for i, split in enumerate(splits):
    counts = [mst.get(split, {}).get(c, 0) for c in CLASSES_ORDER]
    bars = ax.bar(x + i * width, counts, width,
                  label=split.capitalize(), color=split_colors[split],
                  edgecolor='#333', linewidth=0.5)
    for bar, cnt in zip(bars, counts):
        if cnt > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                    f'{cnt:,}', ha='center', va='bottom', fontsize=7, rotation=90)

ax.set_xticks(x + width)
ax.set_xticklabels(short_names, rotation=40, ha='right', fontsize=10)
ax.set_ylabel('Количество тайлов', fontsize=11)
ax.set_title('Распределение классов в датасете (ДС = УФ)', fontsize=13, pad=12)
ax.legend(fontsize=10, loc='upper right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
out = DATASET_STATS / 'class_distribution.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Сохранено: {out}')


# ═══════════════════════════════════════════════════════════════════════════════
# 2. split_table.png — таблица: классы × сплиты с counts и %
# ═══════════════════════════════════════════════════════════════════════════════
rows = []
for cls in CLASSES_ORDER:
    tr = mst.get('train', {}).get(cls, 0)
    vl = mst.get('val',   {}).get(cls, 0)
    te = mst.get('test',  {}).get(cls, 0)
    tot = tr + vl + te
    rows.append([
        CLASS_SHORT.get(cls, cls[:14]),
        f'{tr:,}',
        f'{vl:,}  ({vl/tot*100:.0f}%)' if tot else '0',
        f'{te:,}  ({te/tot*100:.0f}%)' if tot else '0',
        f'{tot:,}',
    ])

# Итого
tr_t = sum(mst.get('train', {}).get(c, 0) for c in CLASSES_ORDER)
vl_t = sum(mst.get('val',   {}).get(c, 0) for c in CLASSES_ORDER)
te_t = sum(mst.get('test',  {}).get(c, 0) for c in CLASSES_ORDER)
tot_t = tr_t + vl_t + te_t
rows.append(['ИТОГО', f'{tr_t:,}', f'{vl_t:,}  ({vl_t/tot_t*100:.0f}%)',
             f'{te_t:,}  ({te_t/tot_t*100:.0f}%)', f'{tot_t:,}'])

col_labels = ['Класс', 'Train', 'Val', 'Test', 'Всего']

fig, ax = plt.subplots(figsize=(13, 5))
ax.axis('off')
tbl = ax.table(
    cellText=rows, colLabels=col_labels,
    cellLoc='center', loc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.0, 1.6)

# Цветим заголовок и итоговую строку
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#1E3A5F')
        cell.set_text_props(color='white', fontweight='bold')
    elif row == len(rows):   # итого
        cell.set_facecolor('#E2E8F0')
        cell.set_text_props(fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#F8FAFC')
    else:
        cell.set_facecolor('white')
    cell.set_edgecolor('#CBD5E1')

ax.set_title('Статистика датасета (тайлы 5 см, ДС = УФ)', fontsize=13, pad=14, fontweight='bold')
plt.tight_layout()
out = DATASET_STATS / 'split_table.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Сохранено: {out}')


# ═══════════════════════════════════════════════════════════════════════════════
# 3. tile_examples.png — сетка: 8 классов × 4 случайных тайла (ДС)
# ═══════════════════════════════════════════════════════════════════════════════
import random as _rnd
_rnd.seed(SEED)

EXAMPLES_PER_CLASS = 4
fig, axes = plt.subplots(
    len(CLASSES_ORDER), EXAMPLES_PER_CLASS,
    figsize=(EXAMPLES_PER_CLASS * 2.2, len(CLASSES_ORDER) * 2.5),
)
fig.suptitle('Примеры тайлов (5 см, ДС) по классам', fontsize=14, fontweight='bold', y=1.01)

for row_i, cls in enumerate(CLASSES_ORDER):
    cls_dir = DS_ROOT / 'ДС' / 'train' / cls
    if not cls_dir.exists():
        cls_dir = DS_ROOT / 'ДС' / 'val' / cls
    files = list(cls_dir.glob('*.jpg')) if cls_dir.exists() else []
    sample = _rnd.sample(files, min(EXAMPLES_PER_CLASS, len(files)))

    # Цветная метка слева
    color = CLASS_PALETTE.get(cls, '#94A3B8')
    short = CLASS_SHORT.get(cls, cls[:10])

    for col_i in range(EXAMPLES_PER_CLASS):
        ax = axes[row_i, col_i]
        if col_i < len(sample):
            img = np.array(Image.open(sample[col_i]).convert('RGB'))
            ax.imshow(img)
        else:
            ax.set_facecolor('#F1F5F9')
        ax.axis('off')
        if col_i == 0:
            ax.set_ylabel(short, rotation=0, labelpad=60, va='center',
                          fontsize=9, fontweight='bold', color=color)

plt.subplots_adjust(wspace=0.05, hspace=0.1)
out = DATASET_STATS / 'tile_examples.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Сохранено: {out}')

print(f'\nВсе артефакты датасета сохранены в: {DATASET_STATS}')